# Ch.12 — Clustering

> **Running theme:** The real estate platform wants to discover natural neighbourhood types — no labels, no target variable. K-Means, DBSCAN, and HDBSCAN each partition the data differently. Together they reveal the hidden structure in California's 20,640 census districts.

## Core Idea

**K-Means:** minimise inertia $J = \sum_k \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$ via alternating assignment → centroid-update steps (Lloyd's algorithm).

**DBSCAN:** density-based — a core point has ≥ `min_samples` neighbours within radius `ε`. Clusters = maximal density-connected sets. Outliers labelled **−1** (noise).

**HDBSCAN:** builds a hierarchy over all density levels via mutual reachability distance + MST. Extracts the most stable clusters. Only needs `min_cluster_size`.

## Running Example

Dataset: **California Housing** (sklearn) — 20,640 census districts, 8 features.  
**No target variable** — pure unsupervised learning.

Question: *What natural neighbourhood types exist in California?* After clustering we will colour the districts by K-Means cluster label to see if income tiers, coastal/inland splits, or density patterns emerge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA          # for 2D visualisation

# ── Load data ──────────────────────────────────────────────────────────────────
housing = fetch_california_housing()
X       = housing.data          # (20640, 8) — no target used
feature_names = list(housing.feature_names)

# ── Scale ─────────────────────────────────────────────────────────────────────
# Distance-based algorithms are scale-sensitive — standardisation is mandatory
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]:,} samples × {X.shape[1]} features")
print("Features:", feature_names)

## K-Means: Elbow Curve

Inertia (within-cluster sum of squares) always decreases as K increases.  
The **elbow** — where marginal gain flattens — suggests the best K.  
Silhouette score (Ch.14) provides a second, independent signal.

In [ ]:
K_range   = range(2, 16)
inertias  = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    # silhouette_score is O(n^2); sample for speed
    sil_scores.append(silhouette_score(X_sc, km.labels_, sample_size=5000, random_state=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_range), inertias, 'b-o', markersize=5)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Curve — inertia vs K')
ax1.grid(True, alpha=0.3)

ax2.plot(list(K_range), sil_scores, 'r-o', markersize=5)
ax2.set_xlabel('K'); ax2.set_ylabel('Mean silhouette score')
ax2.set_title('Silhouette score vs K')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

best_k_sil = list(K_range)[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k_sil}  (score={max(sil_scores):.4f})")

## K-Means: Fit and Interpret Clusters

Fit with the chosen K. Inspect centroids in the **original** (un-scaled) feature space to understand what each cluster represents.

In [ ]:
# ── Fit best K-Means ──────────────────────────────────────────────────────────
best_k = best_k_sil    # use silhouette-based recommendation
km_best = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
km_best.fit(X_sc)
labels_km = km_best.labels_

# ── Decode centroids to original scale ────────────────────────────────────────
centroids_orig = scaler.inverse_transform(km_best.cluster_centers_)

print(f"K = {best_k} cluster profiles (original feature scale)")
print(f"{'Cluster':<9}", "  ".join(f"{n:>10}" for n in feature_names))
for i, c in enumerate(centroids_orig):
    vals = "  ".join(f"{v:>10.2f}" for v in c)
    mean_val = housing.target[labels_km == i].mean()
    n_pts    = (labels_km == i).sum()
    print(f"  C{i:<7} {vals}   | MedHouseVal = {mean_val:.2f}  (n={n_pts:,})")

## K-Means: 2D Visualisation via PCA

We can't plot 8 dimensions directly. Use PCA (Ch.13) to project to 2D and colour by cluster label.

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_sc)
print(f"PCA explained variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Cluster labels ─────────────────────────────────────────────────────────────
sc0 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_km,
                       cmap='tab10', s=2, alpha=0.4)
axes[0].set_title(f'K-Means (K={best_k}) — cluster labels')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc0, ax=axes[0], label='Cluster')

# ── Median house value colour ──────────────────────────────────────────────────
sc1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=housing.target,
                       cmap='viridis', s=2, alpha=0.4)
axes[1].set_title('Actual MedHouseVal')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[1], label='MedHouseVal ($100k)')

plt.tight_layout(); plt.show()

## DBSCAN: ε Estimation via k-NN Distance Plot

**Rule of thumb:** `k = 2 × n_features = 16`.  
Sort all points by their distance to the k-th nearest neighbour.  
The **knee** of the resulting curve is a good starting ε.

In [ ]:
k_nn = 2 * X_sc.shape[1]    # 16
nbrs = NearestNeighbors(n_neighbors=k_nn, algorithm='ball_tree').fit(X_sc)
distances, _ = nbrs.kneighbors(X_sc)
knn_dists = np.sort(distances[:, -1])   # distance to k-th neighbour, sorted ascending

plt.figure(figsize=(10, 4))
plt.plot(knn_dists, lw=1)
plt.xlabel('Points sorted by distance to 16-th neighbour')
plt.ylabel('Distance (standardised space)')
plt.title('k-NN Distance Plot — pick ε at the knee')
plt.axhline(y=0.5, color='r', linestyle='--', label='ε = 0.5 (example)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print(f"25th percentile distance: {np.percentile(knn_dists, 25):.3f}")
print(f"50th percentile distance: {np.percentile(knn_dists, 50):.3f}")
print(f"75th percentile distance: {np.percentile(knn_dists, 75):.3f}")

## DBSCAN: Fit and Inspect

In [ ]:
eps_val      = 0.5    # adjust after inspecting k-NN plot
min_samp_val = 16     # 2 × n_features

db = DBSCAN(eps=eps_val, min_samples=min_samp_val, algorithm='ball_tree', n_jobs=-1)
db.fit(X_sc)
labels_db = db.labels_

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise       = (labels_db == -1).sum()
print(f"DBSCAN (eps={eps_val}, min_samples={min_samp_val})")
print(f"  Clusters found : {n_clusters_db}")
print(f"  Noise points   : {n_noise:,} ({n_noise/len(X_sc)*100:.1f}%)")

# ── Visualise in 2D PCA space ─────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
colours = labels_db.copy().astype(float)
colours[colours == -1] = np.nan    # noise points plotted separately
sc = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=colours, cmap='tab20', s=2, alpha=0.4)
noise_mask = labels_db == -1
plt.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1],
            c='black', s=1, alpha=0.15, label='Noise')
plt.colorbar(sc, label='Cluster')
plt.title(f'DBSCAN — {n_clusters_db} clusters (black = noise)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(markerscale=5); plt.tight_layout(); plt.show()

## HDBSCAN

In [ ]:
try:
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=200, min_samples=10, gen_min_span_tree=False)
    labels_hdb = hdb.fit_predict(X_sc)

    n_clusters_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    n_noise_hdb    = (labels_hdb == -1).sum()
    print(f"HDBSCAN: {n_clusters_hdb} clusters, {n_noise_hdb:,} noise ({n_noise_hdb/len(X_sc)*100:.1f}%)")

    plt.figure(figsize=(8, 5))
    h_colours = labels_hdb.copy().astype(float)
    h_colours[h_colours == -1] = np.nan
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=h_colours, cmap='tab20', s=2, alpha=0.4)
    nm = labels_hdb == -1
    plt.scatter(X_2d[nm, 0], X_2d[nm, 1], c='black', s=1, alpha=0.15, label='Noise')
    plt.title(f'HDBSCAN — {n_clusters_hdb} clusters (black = noise)')
    plt.xlabel('PC1'); plt.ylabel('PC2')
    plt.legend(markerscale=5); plt.tight_layout(); plt.show()

except ImportError:
    print("hdbscan not installed — run: pip install hdbscan")
    print("(Or use: pip install hdbscan --no-build-isolation)")

## Hyperparameter Dial: K-Means K Sweep

Visually compare how cluster boundaries shift as K increases from 2 to 6.

In [ ]:
Ks = [2, 3, 4, 5, 6]
fig, axes = plt.subplots(1, len(Ks), figsize=(18, 4))

for ax, k in zip(axes, Ks):
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    km_k.fit(X_sc)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=km_k.labels_, cmap='tab10', s=1, alpha=0.3)
    ax.set_title(f'K = {k}   inertia={km_k.inertia_:,.0f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.tick_params(labelleft=False, labelbottom=False)

plt.suptitle('K-Means: how cluster boundaries shift with K', y=1.02)
plt.tight_layout(); plt.show()

## Hyperparameter Dial: DBSCAN ε Sweep

In [ ]:
eps_values = [0.2, 0.5, 0.8, 1.5]
fig, axes  = plt.subplots(1, len(eps_values), figsize=(18, 4))

for ax, eps in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps, min_samples=16, algorithm='ball_tree', n_jobs=-1)
    db_e.fit(X_sc)
    lbl   = db_e.labels_
    n_c   = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_nz  = (lbl == -1).sum()
    cols  = lbl.copy().astype(float); cols[cols == -1] = np.nan
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cols, cmap='tab20', s=1, alpha=0.3)
    mask_n = lbl == -1
    ax.scatter(X_2d[mask_n, 0], X_2d[mask_n, 1], c='black', s=0.5, alpha=0.2)
    ax.set_title(f'ε={eps}\n{n_c} clusters, {n_nz/len(X_sc)*100:.0f}% noise')
    ax.tick_params(labelleft=False, labelbottom=False)

plt.suptitle('DBSCAN: ε sweep (too small → all noise; too large → one cluster)', y=1.02)
plt.tight_layout(); plt.show()

## What Can Go Wrong: K-Means on Non-Spherical Data

K-Means assumes clusters are convex and roughly equal-sized. On ring-shaped or elongated distributions it fails completely while DBSCAN succeeds.

In [ ]:
from sklearn.datasets import make_circles, make_moons

np.random.seed(42)
X_circles, _ = make_circles(n_samples=800, factor=0.5, noise=0.05)
X_moons,    _ = make_moons(n_samples=800, noise=0.07)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (Xd, name) in enumerate([(X_circles, 'Circles'), (X_moons, 'Moons')]):
    # K-Means
    km_fail = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xd)
    axes[row, 0].scatter(Xd[:, 0], Xd[:, 1], c=km_fail.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 0].set_title(f'K-Means on {name} — WRONG')

    # DBSCAN
    db_ok = DBSCAN(eps=0.2, min_samples=5).fit(Xd)
    axes[row, 1].scatter(Xd[:, 0], Xd[:, 1], c=db_ok.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 1].set_title(f'DBSCAN on {name} — correct')

plt.suptitle('K-Means fails on non-spherical shapes; DBSCAN succeeds', fontsize=13)
plt.tight_layout(); plt.show()

## What Can Go Wrong: Unscaled Features Distort K-Means

In [ ]:
print("Feature ranges (raw):")
for name, mn, mx in zip(feature_names, X.min(axis=0), X.max(axis=0)):
    print(f"  {name:<14} {mn:8.2f} … {mx:8.2f}")

print("\nAveRooms range is 100× wider than MedInc — unscaled K-Means ignores MedInc entirely.")

km_raw    = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)     # RAW
km_scaled = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_sc)  # SCALED

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(km_scaled.labels_, km_raw.labels_)
print(f"\nARI between raw vs scaled K-Means labels: {ari:.3f}")
print("(ARI=1.0 means identical; ARI≈0 means the clusters are completely different)")

## Exercises

1. **DBSCAN density problem.** The California Housing dataset has dense urban districts (Bay Area, LA) and sparse rural ones. Try `eps=0.3` and `eps=1.0` and explain the difference in number of clusters and noise fraction. Which ε better separates urban from rural?

2. **K-Means++ vs random init.** Run `KMeans(n_clusters=8, init='random', n_init=1)` ten times and record inertia each time. Then run `init='k-means++'`. Plot both inertia distributions as histograms. How much more stable is K-Means++?

3. **HDBSCAN soft clustering.** After fitting HDBSCAN, access `hdb.probabilities_` — the confidence that each point belongs to its assigned cluster. Plot a histogram of these probabilities. What fraction of points are assigned with >90% confidence?

In [ ]:
# Exercise 1 — DBSCAN density problem
# TODO: your solution here

In [ ]:
# Exercise 2 — K-Means++ vs random init
# TODO: your solution here

In [ ]:
# Exercise 3 — HDBSCAN soft clustering probabilities
# TODO: your solution here